In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

print("⚡ Inicializando Motor de Ingesta Structured Streaming (Auto Loader)...")
print("---------------------------------------------------------------------")

# 1. Rutas nativas del Workspace 
ruta_origen_json = "/Workspace/Shared/telemetria_raw/"
ruta_checkpoint_bronce = "/Workspace/Shared/checkpoints/bronce_stream/"

# 2. Limpieza preventiva: Borramos la carpeta de checkpoints vieja por si quedó corrupta
dbutils.fs.rm(ruta_checkpoint_bronce, True)

# 3. El molde o esquema estricto para interpretar los JSON
esquema_json = StructType([
    StructField("id_pozo", StringType(), True),
    StructField("timestamp_sensor", StringType(), True),
    StructField("presion_cabezal_raw", StringType(), True),
    StructField("caudal_bpd_raw", StringType(), True)
])

# 4. LEER EN STREAMING (Auto Loader)
df_streaming_crudo = (spark.readStream
    .format("cloudFiles")               
    .option("cloudFiles.format", "json") 
    .schema(esquema_json)                
    .load(ruta_origen_json))

# 5. AGREGAR COLUMNA DE INGESTA
df_streaming_con_ingesta = df_streaming_crudo.withColumn("fecha_ingesta_bronce", current_timestamp())

# 6. ESCRIBIR EN STREAMING MODO MICRO-BATCH (Hacia la tabla Delta Bronce)
query_stream_bronce = (df_streaming_con_ingesta.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", ruta_checkpoint_bronce) 
    .toTable("control_produccion_pozos.pozos_bronce"))

print("✅ [PROCESO COMPLETADO] Datos crudos ingeridos en la tabla pozos_bronce sin errores.")